# Trabajo Práctico Integrador: Análisis de Desempeño y Gestión de Estudiantes

**Materia:** Análisis de Datos Inicial
**Carrera:** Tecnicatura Universitaria en Programación (TUP)

**Integrantes — Grupo 7:**
1. Jeremías Bontorno Pontis
2. Nicolas Andres Hassan Padoan Vargas
3. Luciano Andres Mas Cannizzo
4. Axel Esteban Mejias
5. Leandro Nicolas Nuñez Agostinho
6. Valentino Vernier

---

## Hito 1: Elección y Planteo

### Dataset elegido: `Calificaciones.csv`

Conjunto de datos académicos de la TUP correspondiente al cuatrimestre marzo–junio 2026. El archivo está en **formato largo**: cada fila representa **una entrega de un alumno en una actividad puntual** (TP, Quiz o Parcial). Esto permite analizar tanto el rendimiento agregado por alumno como los patrones de comportamiento entrega por entrega.

| Característica | Valor |
|---|---|
| Origen | Sistema de gestión académica de la TUP (cuatrimestre 2026-1) |
| Granularidad | 1 fila = 1 entrega de un alumno en una actividad |
| Filas | 5.610 |
| Alumnos únicos | 700 |
| Comisiones | 9 (A1–A3 mañana, B1–B2 tarde, C1–C4 noche) |
| Actividades por alumno | 4 TPs + 3 Quizzes + 1 Parcial = 8 |

**Columnas:** `ID_Alumno`, `Nombre_Apellido`, `Edad`, `Genero`, `Email`, `Comision`, `Turno`, `Fecha_Inscripcion`, `Actividad`, `Tipo_Actividad`, `Fecha_Limite`, `Fecha_Entrega`, `Nota`, `Estado_Entrega`.

### Objetivos del análisis (preguntas a responder)

1. **Patrones de abandono del parcial.** ¿Qué combinación de **constancia** en las entregas (a tiempo, tarde o no realizadas) y **rendimiento previo** en TPs y Quizzes anticipa que un alumno **no se presente al parcial**? ¿Existe un corte en el *Índice de Constancia* que separe claramente a los que rinden del resto?

2. **Diferencias entre comisiones y turnos.** ¿Hay **comisiones o turnos** con tasa de aprobación o nota promedio significativamente menor al resto, y esa diferencia se mantiene al controlar por **tipo de actividad** (TP vs Quiz vs Parcial)?

3. **Evolución temporal del rendimiento.** ¿Cómo evolucionan la **nota promedio** y la **tasa de entrega** a lo largo del cuatrimestre? ¿Existe un **punto de inflexión** (semana o actividad puntual) a partir del cual se concentra el abandono?

In [1]:
# Importación de librerías
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)

RUTA_CSV = os.path.join("..", "data", "Calificaciones.csv")
RUTA_SALIDA = os.path.join("..", "data", "datos_limpios.csv")

---
## Hito 2: ETL y Calidad de Datos

Aplicamos un proceso ETL modularizado en funciones (un paso = una función) para que sea fácil de auditar y de reutilizar desde el dashboard del Hito 4. Cada función:
- recibe un DataFrame y devuelve uno nuevo (no muta el original),
- está envuelta por `try/except` cuando hay riesgo real de fallo (carga de archivo, conversión de tipos),
- imprime un resumen de lo que hizo, para dejar trazabilidad.

**Plan:**
1. Carga + auditoría del CSV crudo.
2. Limpieza: normalización de strings, parseo de notas con coma decimal, parseo de fechas con múltiples formatos, deduplicación.
3. Tratamiento de outliers: rango válido para `Nota` (0–10) e IQR para `Edad`.
4. Feature Engineering: días de anticipación, índice de constancia, promedio previo al parcial, estado final del alumno.
5. Agregación por alumno y exportación a `datos_limpios.csv` (formato que usa el dashboard del Hito 4).

### 2.1 Carga y auditoría inicial

In [2]:
def cargar_dataset(ruta: str) -> pd.DataFrame:
    """Lee el CSV crudo. Eleva un FileNotFoundError con mensaje claro."""
    try:
        df = pd.read_csv(ruta)
    except FileNotFoundError as e:
        raise FileNotFoundError(
            f"No se encontró '{ruta}'. Verificá que el archivo esté "
            f"en la carpeta data/ junto a este notebook."
        ) from e
    print(f"[OK] Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas.")
    return df


def auditar(df: pd.DataFrame) -> None:
    """Imprime tipos, nulos por columna y duplicados exactos."""
    print("--- Tipos de datos ---")
    print(df.dtypes)
    print("\n--- Nulos por columna ---")
    print(df.isna().sum())
    print(f"\n--- Filas duplicadas exactas: {df.duplicated().sum()}")


df_raw = cargar_dataset(RUTA_CSV)
display(df_raw.head())
auditar(df_raw)

[OK] Dataset cargado: 5610 filas x 14 columnas.


,ID_Alumno,Nombre_Apellido,Edad,Genero,Email,Comision,Turno,Fecha_Inscripcion,Actividad,Tipo_Actividad,Fecha_Limite,Fecha_Entrega,Nota,Estado_Entrega
0,A00206,Bautista Medina,17,M,bautista.medina206@alumnos.tup.edu.ar,B2,Tarde,05/03/2026,Quiz_02,Quiz,2026-04-27,26-04-2026,4.1,a tiempo
1,A00363,Lucas Molina,17,Femenino,lucas.molina363@alumnos.tup.edu.ar,B1,Tarde,2026-02-14,Quiz_03,Quiz,25/05/2026,2026-05-22,4.8,A tiempo
2,A00004,Camila Alvarez,23,F,camila.alvarez4@alumnos.tup.edu.ar,C1,Noche,2026-03-06,TP_01,TP,16/03/2026,NaN,NaN,no entregó
3,A00517,Agustin Acosta,22,M,agustin.acosta517@alumnos.tup.edu.ar,A1,Mañana,2026-03-09,Parcial_01,Parcial,2026-06-22,2026-06-27,"7,2",TARDE
4,A00575,Lucas Sanchez,28,M,lucas.sanchez575@alumnos.tup.edu.ar,A3,Mañana,06-03-2026,TP_02,TP,13/04/2026,NaN,NaN,no entregó


--- Tipos de datos ---
ID_Alumno            object
Nombre_Apellido      object
Edad                  int64
Genero               object
Email                object
Comision             object
Turno                object
Fecha_Inscripcion    object
Actividad            object
Tipo_Actividad       object
Fecha_Limite         object
Fecha_Entrega        object
Nota                 object
Estado_Entrega       object
dtype: object

--- Nulos por columna ---
ID_Alumno               0
Nombre_Apellido         0
Edad                    0
Genero                  0
Email                 130
Comision                0
Turno                   0
Fecha_Inscripcion       0
Actividad               0
Tipo_Actividad          0
Fecha_Limite            0
Fecha_Entrega        1646
Nota                 1646
Estado_Entrega        432
dtype: int64

--- Filas duplicadas exactas: 10


### 2.2 Limpieza

Detectamos en la auditoría:
- 14 columnas tipo `object` aunque varias son numéricas o de fecha.
- ~1.646 nulos en `Nota` y `Fecha_Entrega` (legítimos: representan *no entregó*).
- ~432 nulos en `Estado_Entrega` y ~130 en `Email` (campos no críticos).
- 10 filas duplicadas exactas a remover.

Encaramos cada problema con una función dedicada.

In [3]:
def normalizar_strings(df: pd.DataFrame) -> pd.DataFrame:
    """Quita espacios extra y unifica casing en columnas categóricas."""
    out = df.copy()
    out["Nombre_Apellido"] = out["Nombre_Apellido"].astype(str).str.strip().str.title()
    out["Comision"] = out["Comision"].astype(str).str.strip().str.upper()
    out["Turno"] = out["Turno"].astype(str).str.strip().str.title()
    out["Tipo_Actividad"] = out["Tipo_Actividad"].astype(str).str.strip().str.title()
    return out


def unificar_genero(df: pd.DataFrame) -> pd.DataFrame:
    """Reduce las 6 variantes de género a F / M / X / Otro."""
    mapeo = {
        "F": "F", "FEMENINO": "F", "Femenino": "F", "femenino": "F",
        "M": "M", "MASCULINO": "M", "Masculino": "M",
        "X": "X",
    }
    out = df.copy()
    out["Genero"] = (
        out["Genero"].astype(str).str.strip().map(mapeo).fillna("Otro")
    )
    return out


def unificar_estado_entrega(df: pd.DataFrame) -> pd.DataFrame:
    """Reduce las 10 variantes a 3 categorías canónicas."""
    def _mapear(v):
        if pd.isna(v):
            return "No Entrego"
        s = str(v).strip().lower()
        if s == "" or "no" in s:
            return "No Entrego"
        if "tarde" in s:
            return "Tarde"
        if "tiempo" in s:
            return "A Tiempo"
        return "Otro"
    out = df.copy()
    out["Estado_Entrega"] = out["Estado_Entrega"].map(_mapear)
    return out


def parsear_nota(df: pd.DataFrame) -> pd.DataFrame:
    """Convierte 'Nota' a float manejando comas decimales (ej: '7,5')."""
    out = df.copy()
    serie = out["Nota"].astype(str).str.replace(",", ".", regex=False)
    out["Nota"] = pd.to_numeric(serie, errors="coerce")
    return out


def parsear_fechas(df: pd.DataFrame) -> pd.DataFrame:
    """Convierte las 3 columnas de fecha aceptando varios formatos."""
    out = df.copy()
    for col in ["Fecha_Inscripcion", "Fecha_Limite", "Fecha_Entrega"]:
        out[col] = pd.to_datetime(out[col], errors="coerce", format="mixed")
    return out


def remover_duplicados(df: pd.DataFrame) -> pd.DataFrame:
    """Quita filas exactamente duplicadas."""
    antes = len(df)
    out = df.drop_duplicates().reset_index(drop=True)
    print(f"[OK] Duplicados removidos: {antes - len(out)}")
    return out

### 2.3 Tratamiento de outliers (método estadístico)

Aplicamos dos enfoques distintos según la variable:
- **`Nota`**: tiene un rango definido por la cátedra (0 a 10). Cualquier valor fuera de ese rango es un error de carga, lo marcamos como `NaN`.
- **`Edad`**: usamos el método de **IQR (rango intercuartílico)**. Calculamos Q1 y Q3, definimos el límite válido como `[Q1 − 1.5·IQR, Q3 + 1.5·IQR]` (acotado a 16–70) e imputamos con la mediana los valores fuera de ese rango.

In [4]:
def tratar_outliers_nota(df: pd.DataFrame) -> pd.DataFrame:
    """Notas fuera de [0, 10] -> NaN."""
    out = df.copy()
    mascara = (out["Nota"] < 0) | (out["Nota"] > 10)
    n = int(mascara.sum())
    out.loc[mascara, "Nota"] = np.nan
    print(f"[OK] Notas fuera de rango (0-10) marcadas como NaN: {n}")
    return out


def tratar_outliers_edad(df: pd.DataFrame) -> pd.DataFrame:
    """Imputa edades fuera del IQR con la mediana."""
    out = df.copy()
    edades_validas = out.loc[(out["Edad"] >= 16) & (out["Edad"] <= 70), "Edad"]
    q1, q3 = edades_validas.quantile([0.25, 0.75])
    iqr = q3 - q1
    limite_inf = max(16, q1 - 1.5 * iqr)
    limite_sup = min(70, q3 + 1.5 * iqr)
    mediana = edades_validas.median()
    mascara = (out["Edad"] < limite_inf) | (out["Edad"] > limite_sup)
    n = int(mascara.sum())
    out.loc[mascara, "Edad"] = mediana
    print(
        f"[OK] Edades fuera de IQR ({limite_inf:.0f}-{limite_sup:.0f}) "
        f"imputadas con la mediana ({mediana:.0f}): {n}"
    )
    return out

### 2.4 Aplicación del pipeline ETL

Encadenamos las funciones en un orquestador. Si en el futuro queremos agregar o quitar pasos, solo modificamos la lista.

In [5]:
def aplicar_etl(df: pd.DataFrame) -> pd.DataFrame:
    """Ejecuta secuencialmente todas las funciones de limpieza."""
    pasos = [
        normalizar_strings,
        unificar_genero,
        unificar_estado_entrega,
        parsear_nota,
        parsear_fechas,
        remover_duplicados,
        tratar_outliers_nota,
        tratar_outliers_edad,
    ]
    out = df
    for paso in pasos:
        out = paso(out)
    return out


df_limpio = aplicar_etl(df_raw)

print("\n--- Tipos finales ---")
print(df_limpio.dtypes)
print(f"\nFilas: {len(df_limpio)}")

[OK] Duplicados removidos: 10
[OK] Notas fuera de rango (0-10) marcadas como NaN: 0
[OK] Edades fuera de IQR (16-36) imputadas con la mediana (22): 64

--- Tipos finales ---
ID_Alumno                    object
Nombre_Apellido              object
Edad                          int64
Genero                       object
Email                        object
Comision                     object
Turno                        object
Fecha_Inscripcion    datetime64[ns]
Actividad                    object
Tipo_Actividad               object
Fecha_Limite         datetime64[ns]
Fecha_Entrega        datetime64[ns]
Nota                        float64
Estado_Entrega               object
dtype: object

Filas: 5600


### 2.5 Feature Engineering

Creamos variables derivadas pensadas para responder las 3 preguntas:

| Nueva variable | Cómo se calcula | Para qué sirve |
|---|---|---|
| `Dias_Anticipacion` | `Fecha_Limite − Fecha_Entrega` (días) | Mide si el alumno entrega con tiempo, sobre la fecha o tarde. |
| `Hizo_Entrega` | 1 si `Nota` no es nulo, 0 si no | Base para el Índice de Constancia. |
| `Entrega_Aprobada` | 1 si `Nota >= 6` | Permite mirar tasas de aprobación por actividad. |
| `Indice_Constancia` (por alumno) | `% de actividades previas al parcial entregadas a tiempo` | **Variable estrella** para la pregunta 1. |
| `Promedio_Previo` (por alumno) | Media de `Nota` en TPs y Quizzes (sin parcial) | Predictor de comportamiento futuro. |
| `Estado_Final` (por alumno) | Regla sobre nota del parcial: ≥6 Aprobado, 4–6 Recupera, <4 Desaprobado, NaN Ausente | Etiqueta para el dashboard. |

In [6]:
def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """Variables derivadas a nivel fila (entrega)."""
    out = df.copy()
    out["Dias_Anticipacion"] = (out["Fecha_Limite"] - out["Fecha_Entrega"]).dt.days
    out["Hizo_Entrega"] = out["Nota"].notna().astype(int)
    out["Entrega_Aprobada"] = (out["Nota"] >= 6).astype("Int64")
    return out


df_limpio = feature_engineering(df_limpio)
df_limpio.head()

,ID_Alumno,Nombre_Apellido,Edad,Genero,Email,Comision,Turno,Fecha_Inscripcion,Actividad,Tipo_Actividad,Fecha_Limite,Fecha_Entrega,Nota,Estado_Entrega,Dias_Anticipacion,Hizo_Entrega,Entrega_Aprobada
0,A00206,Bautista Medina,17,M,bautista.medina206@alumnos.tup.edu.ar,B2,Tarde,2026-05-03,Quiz_02,Quiz,2026-04-27,2026-04-26,4.1,A Tiempo,1.0,1,0
1,A00363,Lucas Molina,17,F,lucas.molina363@alumnos.tup.edu.ar,B1,Tarde,2026-02-14,Quiz_03,Quiz,2026-05-25,2026-05-22,4.8,A Tiempo,3.0,1,0
2,A00004,Camila Alvarez,23,F,camila.alvarez4@alumnos.tup.edu.ar,C1,Noche,2026-03-06,TP_01,Tp,2026-03-16,NaT,NaN,No Entrego,NaN,0,0
3,A00517,Agustin Acosta,22,M,agustin.acosta517@alumnos.tup.edu.ar,A1,Mañana,2026-03-09,Parcial_01,Parcial,2026-06-22,2026-06-27,7.2,Tarde,-5.0,1,1
4,A00575,Lucas Sanchez,28,M,lucas.sanchez575@alumnos.tup.edu.ar,A3,Mañana,2026-06-03,TP_02,Tp,2026-04-13,NaT,NaN,No Entrego,NaN,0,0


### 2.6 Agregación por alumno + exportación

El dashboard del Hito 4 consume **una fila por alumno**. Acá hacemos esa transformación: agrupamos por `ID_Alumno`, calculamos las métricas derivadas y guardamos `datos_limpios.csv`.

In [7]:
def agregar_por_alumno(df: pd.DataFrame) -> pd.DataFrame:
    """Construye la tabla por alumno (1 fila = 1 alumno) con KPIs."""
    df = df.copy()

    previas = df[df["Tipo_Actividad"].isin(["Tp", "Quiz"])]
    constancia = previas.groupby("ID_Alumno").agg(
        Entregas_Previas=("Hizo_Entrega", "sum"),
        Entregas_A_Tiempo=("Estado_Entrega", lambda s: (s == "A Tiempo").sum()),
        Total_Actividades_Previas=("Hizo_Entrega", "size"),
        Promedio_Previo=("Nota", "mean"),
    )
    constancia["Indice_Constancia"] = (
        100.0 * constancia["Entregas_A_Tiempo"]
        / constancia["Total_Actividades_Previas"]
    ).round(2)

    parcial = (
        df[df["Tipo_Actividad"] == "Parcial"]
        .groupby("ID_Alumno")["Nota"].first().rename("Nota_Parcial")
    )

    fijos = df.groupby("ID_Alumno").agg(
        Nombre_Apellido=("Nombre_Apellido", "first"),
        Edad=("Edad", "first"),
        Genero=("Genero", "first"),
        Comision=("Comision", "first"),
        Turno=("Turno", "first"),
    )

    alumno = fijos.join(constancia).join(parcial).reset_index()

    def _estado(fila):
        n = fila["Nota_Parcial"]
        if pd.isna(n):
            return "Ausente"
        if n >= 6:
            return "Aprobado"
        if n >= 4:
            return "Recupera"
        return "Desaprobado"
    alumno["Estado_Final"] = alumno.apply(_estado, axis=1)

    # Aliases para que el dashboard del Hito 4 los pueda consumir directo.
    alumno["Nota Final"] = alumno["Nota_Parcial"]
    alumno["Estado"] = alumno["Estado_Final"]
    alumno["Comisión"] = alumno["Comision"]
    return alumno


df_alumno = agregar_por_alumno(df_limpio)
df_alumno.head()

,ID_Alumno,Nombre_Apellido,Edad,Genero,Comision,Turno,Entregas_Previas,Entregas_A_Tiempo,Total_Actividades_Previas,Promedio_Previo,Indice_Constancia,Nota_Parcial,Estado_Final,Nota Final,Estado,Comisión
0,A00001,Catalina Ortiz,25,M,C1,Noche,6,5,7,7.483333,71.43,6.7,Aprobado,6.7,Aprobado,C1
1,A00002,Mora Nuñez,24,F,B2,Tarde,5,2,7,4.820000,28.57,5.1,Recupera,5.1,Recupera,B2
2,A00003,Mateo Benitez,25,F,B2,Tarde,5,4,7,6.120000,57.14,9.0,Aprobado,9.0,Aprobado,B2
3,A00004,Camila Alvarez,23,F,C1,Noche,4,3,7,6.125000,42.86,NaN,Ausente,NaN,Ausente,C1
4,A00005,Thiago Luna,31,M,B2,Tarde,7,6,7,8.028571,85.71,8.5,Aprobado,8.5,Aprobado,B2


In [8]:
try:
    df_alumno.to_csv(RUTA_SALIDA, index=False, encoding="utf-8")
    print(f"[OK] Exportado: {RUTA_SALIDA} ({len(df_alumno)} filas)")
except OSError as e:
    print(f"[ERROR] No se pudo escribir el CSV: {e}")

[OK] Exportado: ../data/datos_limpios.csv (700 filas)


### 2.7 Verificación final del ETL

In [9]:
print(f"Alumnos únicos:        {df_alumno['ID_Alumno'].nunique()}")
print(f"Comisiones distintas:  {sorted(df_alumno['Comision'].unique())}")
print(f"Turnos distintos:      {sorted(df_alumno['Turno'].unique())}")
print(f"Estados finales:       {df_alumno['Estado_Final'].value_counts().to_dict()}")
print(f"Tasa entrega parcial:  {100*df_alumno['Nota_Parcial'].notna().mean():.1f}%")
print(f"Promedio Nota Parcial: {df_alumno['Nota_Parcial'].mean():.2f}")
print(f"Mediana Indice_Const.: {df_alumno['Indice_Constancia'].median():.1f}%")

Alumnos únicos:        700
Comisiones distintas:  ['A1', 'A2', 'A3', 'B1', 'B2', 'C1', 'C2', 'C3', 'C4']
Turnos distintos:      ['Mañana', 'Noche', 'Tarde']
Estados finales:       {'Aprobado': 309, 'Ausente': 281, 'Recupera': 96, 'Desaprobado': 14}
Tasa entrega parcial:  59.9%
Promedio Nota Parcial: 7.00
Mediana Indice_Const.: 57.1%


---
## Hito 3: Análisis y Visualización

*Pendiente.*

## Hito 4: Dashboard Interactivo

*Pendiente.*

## Hito 5: Informe de Gestión

*Pendiente.*